In [0]:
import yfinance as yf
import pandas as pd
ticker_lst =['PL=F', 'GC=F','SI=F','HG=F','PA=F']
dt = yf.download(ticker_lst, start='2022-01-01', group_by='ticker')
#Download historical data for the last year
df = pd.DataFrame(dt)
df_f = df.reset_index()
df_f.head(10)
df_f.columns =  ['_'.join(col).strip() for col in df_f.columns.values]
df_f.columns = ["".join(col).replace('=','_') for col in df_f.columns.values]
df_f.update
df_s = spark.createDataFrame(df_f).write.mode('overwrite').saveAsTable("futures.Commodity_test")
display(df_s)

/home/spark-4b769269-2d98-4e1d-aafb-3b/.ipykernel/3549/command-5717214190126703-1310071265:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  dt = yf.download(ticker_lst, start='2022-01-01', group_by='ticker')
[*********************100%***********************]  5 of 5 completed


# Test Partition by Year below

In [0]:
#Join example
from pyspark.sql import SparkSession
from pyspark.sql.functions import year, to_date, try_to_date, col
spark.conf.get("spark.databricks.clusterUsageTags.clusterId")
df_gld_slv = spark.read.table("futures.adf_run_2").select("Date_", "SI_F_Close", "GC_F_Close")
df_gld_slv.withColumn("GLD_SLV_Ration", col("SI_F_Close")/col("GC_F_Close"))
df_bls = spark.read.table("futures.bls")
df_bls_f = df_bls.withColumn("Dt", try_to_date(col("Dt"), "M/d/yyyy"))
df_jn = df_gld_slv.join(df_bls, df_gld_slv.Date_==df_bls.Dt, "left")
df_jn.write.mode("overwrite").format("delta").saveAsTable("futures.bls_gold_silver")

#Read from deltalake path in parquet format
from delta.tables import DeltaTable
from pyspark.sql.functions import *
path = "abfss://bronze@greedystore.dfs.core.windows.net/futures/"
#df = spark.read.format("delta").option("versionAsOf",4).table("futures.job_test")
#df = spark.read.format("delta").option("timestampAsOf","2025-11-25 11:04:00").table("futures.job_test")
df = spark.read.format("delta").table("futures.job_test")
display(df)
df_vol = df.groupBy("Yr").agg(sum("GC_F_Volume"))
display(df_vol.show())
#df_test = df_test.withColumn("Yr",year(col("Date_")) )
#df_test.write.mode("overwrite").partitionBy("Yr").save("abfss://bronze@greedystore.dfs.core.windows.net/futures")
display(df_test.select("Date_", "SI_F_Close", "Yr").distinct().show())
df_test.p



Date_,SI_F_Open,SI_F_High,SI_F_Low,SI_F_Close,SI_F_Volume,GC_F_Open,GC_F_High,GC_F_Low,GC_F_Close,GC_F_Volume,exe_timeStamp,num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows,Yr
2021-01-04,27.284000396728516,27.284000396728516,27.284000396728516,27.284000396728516,52,1912.199951171875,1945.0999755859375,1912.199951171875,1944.699951171875,154,2025-11-24T01:04:27.444Z,null,null,null,null,2021
2021-01-05,27.570999145507812,27.570999145507812,27.570999145507812,27.570999145507812,4,1941.699951171875,1952.699951171875,1941.300048828125,1952.699951171875,113,2025-11-24T01:04:27.444Z,null,null,null,null,2021
2021-01-06,27.424999237060547,27.424999237060547,26.972999572753906,26.972999572753906,50,1952.0,1959.9000244140625,1901.5,1906.9000244140625,331,2025-11-24T01:04:27.444Z,null,null,null,null,2021
2021-01-07,27.200000762939453,27.200000762939453,27.200000762939453,27.200000762939453,3,1922.5999755859375,1926.699951171875,1912.0,1912.300048828125,122,2025-11-24T01:04:27.444Z,null,null,null,null,2021
2021-01-08,24.582000732421875,25.325000762939453,24.582000732421875,24.582000732421875,3,1908.0,1908.0,1834.0999755859375,1834.0999755859375,60,2025-11-24T01:04:27.444Z,null,null,null,null,2021
2021-01-11,25.520000457763672,25.520000457763672,24.725000381469727,25.23900032043457,11,1826.5,1849.5999755859375,1826.5,1849.5999755859375,20,2025-11-24T01:04:27.444Z,null,null,null,null,2021
2021-01-12,25.139999389648438,25.385000228881836,25.104999542236328,25.384000778198242,90,1849.0,1859.699951171875,1839.4000244140625,1842.9000244140625,176,2025-11-24T01:04:27.444Z,null,null,null,null,2021
2021-01-13,25.545000076293945,25.545000076293945,25.5,25.520000457763672,211,1858.0,1858.5999755859375,1853.5999755859375,1853.5999755859375,189,2025-11-24T01:04:27.444Z,null,null,null,null,2021
2021-01-14,25.746000289916992,25.746000289916992,25.746000289916992,25.746000289916992,66,1836.699951171875,1850.699951171875,1836.699951171875,1850.300048828125,149,2025-11-24T01:04:27.444Z,null,null,null,null,2021
2021-01-15,25.43000030517578,25.469999313354492,24.825000762939453,24.825000762939453,88,1829.4000244140625,1829.4000244140625,1825.0,1829.300048828125,31,2025-11-24T01:04:27.444Z,null,null,null,null,2021


+----+----------------+
|  Yr|sum(GC_F_Volume)|
+----+----------------+
|NULL|         5375423|
|2021|         1172775|
|2024|         1065885|
|2023|          996487|
|2022|         1004662|
|2025|         1135614|
+----+----------------+



In [0]:
from pyspark.sql.functions import *


df = spark.read.table("futures.bls").where(year("Dt")==year(current_date()))
df = df.withColumn("Month", split(df.Dt, "-").getItem(1))
display(df.show())
#df = df.filter(col("Dt")>'2025-05-01')
labor_force_agg = df.filter(col("Series_Nm")=="Labor").groupBy(year(col("Dt"))).agg(sum("Val"), mean("Val"), max("Val"))
display(labor_force_agg.show())
df = df.withColumn("CPI", when(df.Series_Nm=="Labor", "Labor_Force").otherwise("CPI"))
display(df.select("CPI").distinct().show())
#display(df.select("Month").distinct().orderBy("Month", asc=True).show())
#display(df.orderBy('Dt', desc=True))
#max_dt = df.select(max('Dt')).alias('max_dt')
#display(max_dt.show())
#last_3_cd = df.select(substr(df.Series_ID, length(df.Series_ID)-2, length(df.Series_ID))).distinct()
display(last_3_cd.show())

+-------------+----------+--------+----------+-------+--------+-------+---+---------+--------+--------+--------+-----+
|    Series_ID|        Dt|     val|Per_Change|Monhtly|3_months|Half_Yr|YoY|Series_Nm| datekey|  mmyyyy|CPI_FLAG|Month|
+-------------+----------+--------+----------+-------+--------+-------+---+---------+--------+--------+--------+-----+
|  LNS11000000|2025-08-01|170778.0|       1.4|    0.3|     0.2|    0.2|1.4|    Labor|20250801|20250801|       0|   08|
|  LNS11000000|2025-07-01|170342.0|       1.2|    0.0|    -0.5|   -0.2|1.2|    Labor|20250701|20250701|       0|   07|
|  LNS11000000|2025-06-01|170380.0|       1.4|   -0.1|    -0.1|    1.1|1.4|    Labor|20250601|20250601|       0|   06|
|  LNS11000000|2025-05-01|170510.0|       1.6|   -0.4|     0.1|    1.3|1.6|    Labor|20250501|20250501|       0|   05|
|  LNS11000000|2025-04-01|171135.0|       1.9|    0.3|     0.2|    1.6|1.9|    Labor|20250401|20250401|       0|   04|
|  LNS11000000|2025-03-01|170591.0|       1.6|  

In [0]:
import yfinance as yf
import pandas as pd 
import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as f

ticker_lst =['GC=F','SI=F']
dt = yf.download(ticker_lst, start='2021-01-01', group_by='ticker')
#Download historical data for the last year
df = pd.DataFrame(dt)
df_f = df.reset_index()
df_f.head(10)
df_f.columns = ['_'.join(col).strip() for col in df_f.columns.values]
df_f.columns = ["".join(col).replace('=', '_') for col in df_f.columns.values]
df_f['Date'] = pd.to_datetime(df_f['Date_'], format='%y-%M-%d')
#create spark dataframe out of yfinance pandas 
df_new = spark.createDataFrame(df_f)
df_new = df_new.withColumn("exe_timeStamp", f.current_timestamp())
df_new = df_new.withColumn("Date_", f.to_date(df_new.Date_,'yyyy-MM-dd'))
#read current data
df_current = DeltaTable.forName(spark, 'futures.staging_yf')
#merge Delta table with new data
df_upsert = df_current.alias("old").merge(df_new.alias("new"), "new.Date_ = old.Date_")\
    .whenNotMatchedInsert(values = {'Date_': 'new.Date_', 'SI_F_Close': 'new.SI_F_Close', 'GC_F_Close': 'new.GC_F_Close', 'SI_F_Open': 'new.SI_F_Open', 'SI_F_High': 'new.SI_F_High', 'SI_F_Low': 'new.SI_F_Low', 'SI_F_Volume': 'new.SI_F_Volume', 'GC_F_Open': 'new.GC_F_Open', 'GC_F_High': 'new.GC_F_High', 'GC_F_Low': 'new.GC_F_Low', 'GC_F_Volume': 'new.GC_F_Volume', 'exe_timeStamp': 'new.exe_timeStamp'}).execute()
display(df_upsert)
df_upsert.write.mode("append").format("delta").option("mergeSchema", "true").saveAsTable("futures.staging_yf")

[*********************100%***********************]  2 of 2 completed


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
import yfinance as yf
import pandas as pd
from delta.tables import DeltaTable
from pyspark.sql.functions import year, to_date, try_to_date 
#create merge

#begin to download new data
ticker_lst =['GC=F','SI=F']
dt = yf.download(ticker_lst, start='2022-01-01', group_by='ticker')
#Download historical data for the last year
df = pd.DataFrame(dt)
df_f = df.reset_index()
df_f.head(10)
df_f.columns =  ['_'.join(col).strip() for col in df_f.columns.values]
df_f.columns = ["".join(col).replace('=','_') for col in df_f.columns.values]
df_f["Date_"] = pd.to_datetime(df_f["Date_"], format="%Y-%m-%d")
df_f.head(10)
#read delta table forName with spark session
df_current = DeltaTable.forName(spark, "futures.adf_run_2")
#merge Delta table with new data
df_upsert = df_current.alias("current").merge(spark.createDataFrame(df_f).alias("new"),"current.Date_=new.Date_").whenMatchedUpdate(
    set = { "SI_F_Close": "new.SI_F_Close","GC_F_Close": "new.GC_F_Close", "Date_": "new.Date_", "SI_F_Open": "new.SI_F_Open",
            "SI_F_High": "new.SI_F_High", "SI_F_Low": "new.SI_F_Low", "SI_F_Volume": "new.SI_F_Volume",
             "GC_F_Open": "new.GC_F_Open", "GC_F_High": "new.GC_F_High",  "GC_F_Low": "new.GC_F_Low", "GC_F_Volume": "new.GC_F_Volume"}
    ).whenNotMatchedInsert(values={'Date_': 'new.Date_', 'SI_F_Close': 'new.SI_F_Close', 'GC_F_Close': 'new.GC_F_Close', 'SI_F_Open': 'new.SI_F_Open', 'SI_F_High': 'new.SI_F_High', 'SI_F_Low': 'new.SI_F_Low', 'SI_F_Volume': 'new.SI_F_Volume', 'GC_F_Open': 'new.GC_F_Open', 'GC_F_High': 'new.GC_F_High', 'GC_F_Low': 'new.GC_F_Low', 'GC_F_Volume': 'new.GC_F_Volume'}).execute()
df_upsert.toPandas().head(10)
df_upsert.write.mode("append").format("delta").option("mergeSchema", "true").saveAsTable("futures.adf_run_2").l

/home/spark-2cfb4fa1-5140-4b41-a28f-c6/.ipykernel/18356/command-5386507204292033-1550179379:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  dt = yf.download(ticker_lst, start='2022-01-01', group_by='ticker')
[*********************100%***********************]  2 of 2 completed


,num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,979,978,0,1


Below is to load and test merge best practices. 

In [0]:
import yfinance as yf
import pandas as pd 
import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions

ticker_lst =['GC=F','SI=F']
dt = yf.download(ticker_lst, start='2021-01-01', group_by='ticker')
#Download historical data for the last year
df = pd.DataFrame(dt)
df_f = df.reset_index()
df_f.head(10)
df_f.columns = ['_'.join(col).strip() for col in df_f.columns.values]
df_f.columns = ["".join(col).replace('=', '_') for col in df_f.columns.values]
#df_f['Date_'] = pd.to_datetime(df_f['Date_'], format='%y-%M-%d')
#create spark dataframe out of yfinance pandas 
df_staging = spark.createDataFrame(df_f)
df_staging = df_staging.withColumn("Date_", f.to_date(df_staging.Date_,'yyyy-MM-dd'))
df_staging = df_staging.withColumn("exe_timeStamp", f.current_timestamp())
df_staging.write.mode("overwrite").format("delta").saveAsTable("futures.job_test")


#remove duplicates
df_dup = spark.read.table("futures.staging_yf")
df_drop_dup = df_dup.dropDuplicates(['Date_'])
df_drop_dup.write.mode("overwrite").format("delta").saveAsTable("futures.staging_yf")

from delta.tables import DeltaTable
df = DeltaTable.forName(spark, 'futures.staging_yf').toDF
df = df.where(year("Date_")==2025)
display(df)
df_blob = spark.read.format("csv").option("header", "true").load("abfss://bronze@greedystore.dfs.core.windows.net/Commodity_2020.csv")
display(df_blob.show())

files =  [f for f in dbutils.fs.ls("abfss://bronze@greedystore.dfs.core.windows.net/")]
display(files)
f = spark.createDataFrame(files)
display(df.head())

/home/spark-b2430987-d48f-4531-a56b-34/.ipykernel/6566/command-5078264166769810-1035502152:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  dt = yf.download(ticker_lst, start='2021-01-01', group_by='ticker')
[*********************100%***********************]  2 of 2 completed
